# Lab 8: Real-Time Stream Processing

A velocity check — flag an account with 3 or more transfers in a 20-second
window, the same basic pattern real fraud/AML systems use, running
continuously against Atlas as new transfers arrive. No app code hosting a
listener loop, no separate stream-processing cluster to run yourself: Atlas
Stream Processing is a managed service that runs the aggregation pipeline
for you.

**Requires an Atlas API key** (Project Owner or Project Stream Processing
Owner role) — set `ATLAS_PUBLIC_KEY` / `ATLAS_PRIVATE_KEY` /
`ATLAS_PROJECT_ID` / `ATLAS_CLUSTER_NAME` in `.env`.

**This has no free tier.** Every second a stream processor is in the
`STARTED` state, it's billed (SP10 tier by default here). This notebook
stops the processor at the end — don't skip that cell if you're re-running
interactively.


In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pymongo[encryption]", "certifi", "python-dotenv", "requests", "matplotlib", "pandas"],
        check=True,
    )
    from getpass import getpass
    ATLAS_URI = os.environ.get("ATLAS_URI") or getpass("Atlas connection string (ATLAS_URI): ")
else:
    from dotenv import load_dotenv
    load_dotenv()
    ATLAS_URI = os.environ["ATLAS_URI"]

DEMO_DB = os.environ.get("DEMO_DB", "banking_demo")
import certifi
CA_FILE = certifi.where()
print("Environment:", "Colab" if IN_COLAB else "local", "| DB:", DEMO_DB)

ATLAS_PUBLIC_KEY = os.environ["ATLAS_PUBLIC_KEY"]
ATLAS_PRIVATE_KEY = os.environ["ATLAS_PRIVATE_KEY"]
ATLAS_PROJECT_ID = os.environ["ATLAS_PROJECT_ID"]
ATLAS_CLUSTER_NAME = os.environ.get("ATLAS_CLUSTER_NAME", "Cluster0")

import time, random
from requests.auth import HTTPDigestAuth
import requests

BASE = "https://cloud.mongodb.com/api/atlas/v2"
AUTH = HTTPDigestAuth(ATLAS_PUBLIC_KEY, ATLAS_PRIVATE_KEY)

def atlas_call(method, path, version, body=None):
    """Atlas Admin API call. Each resource type here pins its own API
    version (Accept always, Content-Type when sending a body) — Stream
    Processing instances/connections use 2023-02-01, processors use
    2024-05-30. Mixing them up returns a 400, not a helpful error."""
    content_type = f"application/vnd.atlas.{version}+json"
    headers = {"Accept": content_type}
    if body is not None:
        headers["Content-Type"] = content_type
    resp = requests.request(method, f"{BASE}{path}", auth=AUTH, headers=headers, json=body)
    resp.raise_for_status()
    return resp.json() if resp.text else None


## Set up the workspace and connection

A Stream Processing **workspace** is where processors run — pick a region
close to your cluster to keep latency down. A **connection** registers
your Atlas cluster as something a processor can read from or write to; both
of these are one-time setup and safe to re-run (they're skipped if they
already exist).


In [ ]:
WORKSPACE = "banking-demo-streams"
CONNECTION = "bankingCluster"

existing = atlas_call("GET", f"/groups/{ATLAS_PROJECT_ID}/streams", "2023-02-01")
if WORKSPACE not in [w["name"] for w in existing.get("results", [])]:
    atlas_call("POST", f"/groups/{ATLAS_PROJECT_ID}/streams", "2023-02-01", {
        "name": WORKSPACE,
        "dataProcessRegion": {"cloudProvider": "GCP", "region": "US_CENTRAL1"},
        "streamConfig": {"tier": "SP10"},
    })
    print(f"Created workspace {WORKSPACE!r}. Give it ~15s before creating a processor on it.")
    time.sleep(15)
else:
    print(f"Workspace {WORKSPACE!r} already exists — reusing it.")

conns = atlas_call("GET", f"/groups/{ATLAS_PROJECT_ID}/streams/{WORKSPACE}/connections", "2023-02-01")
if CONNECTION not in [c["name"] for c in conns.get("results", [])]:
    atlas_call("POST", f"/groups/{ATLAS_PROJECT_ID}/streams/{WORKSPACE}/connections", "2023-02-01", {
        "name": CONNECTION,
        "type": "Cluster",
        "clusterName": ATLAS_CLUSTER_NAME,
        "dbRoleToExecute": {"role": "readWriteAnyDatabase", "type": "BUILT_IN"},
    })
    print(f"Created connection {CONNECTION!r} -> cluster {ATLAS_CLUSTER_NAME!r}.")
else:
    print(f"Connection {CONNECTION!r} already exists — reusing it.")


## The pipeline

A stream processor is an aggregation pipeline, same syntax you already know
— it just runs continuously instead of once:

- `$source` watches `banking_demo.stream_transfers` as a change stream.
- `$match` keeps only inserts (a change stream also emits updates/deletes).
- `$tumblingWindow` batches 20 seconds of events at a time, then runs a
  `$group` + `$match` inside it to find accounts with 3+ transfers in that
  window.
- `$merge` writes anything that trips the check into `fraud_alerts`.

One detail worth knowing if you write your own: `boundary: "processingTime"`
below means windows close on wall-clock time. Leave it out and windows
close based on the *event* timestamps flowing through the pipeline instead
— which stalls forever if the source goes idle for longer than the window
size, since there's nothing left to advance the clock. For a live demo
where traffic starts and stops, processing-time windows are the ones that
actually close on schedule.


In [ ]:
PROCESSOR = "velocityCheck"

pipeline = [
    {"$source": {
        "connectionName": CONNECTION,
        "db": DEMO_DB,
        "coll": "stream_transfers",
        "config": {"fullDocument": "updateLookup"},
    }},
    {"$match": {"operationType": "insert"}},
    {"$tumblingWindow": {
        "boundary": "processingTime",
        "interval": {"size": 20, "unit": "second"},
        "pipeline": [
            {"$group": {
                "_id": "$fullDocument.account_id",
                "transfer_count": {"$sum": 1},
                "total_amount": {"$sum": "$fullDocument.amount"},
                "transfer_ids": {"$push": "$fullDocument.transfer_id"},
                "window_start": {"$first": {"$meta": "stream.window.start"}},
                "window_end": {"$first": {"$meta": "stream.window.end"}},
            }},
            {"$match": {"transfer_count": {"$gte": 3}}},
        ],
    }},
    {"$addFields": {"alert_type": "velocity_check"}},
    {"$merge": {"into": {"connectionName": CONNECTION, "db": DEMO_DB, "coll": "fraud_alerts"}}},
]

procs = atlas_call("GET", f"/groups/{ATLAS_PROJECT_ID}/streams/{WORKSPACE}/processors", "2024-05-30")
if PROCESSOR not in [p["name"] for p in procs.get("results", [])]:
    atlas_call("POST", f"/groups/{ATLAS_PROJECT_ID}/streams/{WORKSPACE}/processor", "2024-05-30", {
        "name": PROCESSOR,
        "pipeline": pipeline,
        # A dead letter queue catches documents the pipeline can't process
        # (bad shape, type mismatch) instead of silently dropping them.
        "options": {"dlq": {"connectionName": CONNECTION, "db": DEMO_DB, "coll": "stream_dlq"}},
    })
    print(f"Created processor {PROCESSOR!r}.")
else:
    print(f"Processor {PROCESSOR!r} already exists — reusing it.")

status = atlas_call("GET", f"/groups/{ATLAS_PROJECT_ID}/streams/{WORKSPACE}/processor/{PROCESSOR}", "2024-05-30")
print("Current state:", status["state"])


## Start it, then generate some traffic

Billing starts the moment this next cell runs.


In [ ]:
if status["state"] != "STARTED":
    atlas_call("POST", f"/groups/{ATLAS_PROJECT_ID}/streams/{WORKSPACE}/processor/{PROCESSOR}:start", "2024-05-30")
    print("Processor starting...")
    time.sleep(8)  # let the change stream attach before we start writing
else:
    print("Already running.")


In [ ]:
from pymongo import MongoClient

client = MongoClient(ATLAS_URI, tlsCAFile=CA_FILE)
db = client[DEMO_DB]
db.stream_transfers.drop()
db.fraud_alerts.drop()

def make_transfer(i, account_id, amount):
    return {"transfer_id": f"ST{i:06d}", "account_id": account_id, "amount": amount}

print("Inserting a few ordinary, unrelated transfers...")
for i in range(3):
    db.stream_transfers.insert_one(make_transfer(i, f"ACC{1000+i}", round(random.uniform(50, 500), 2)))
    time.sleep(1)

print("Inserting a burst of transfers from ACC9999 — this should trip the velocity check...")
for i in range(4):
    db.stream_transfers.insert_one(make_transfer(100 + i, "ACC9999", round(random.uniform(2000, 9000), 2)))


In [ ]:
print("Waiting for the 20-second window to close...")
deadline = time.time() + 40
alert = None
while time.time() < deadline:
    alert = db.fraud_alerts.find_one({"_id": "ACC9999"})
    if alert:
        break
    time.sleep(2)

if alert:
    print(f"Flagged: {alert['transfer_count']} transfers, "
          f"${alert['total_amount']:,.2f} total, account {alert['_id']}")
    print(alert)
else:
    print("No alert yet — tumbling windows are wall-clock aligned, so if the burst landed")
    print("right at a window boundary, part of it may show up in the next window instead.")
    print("Re-run the cell above, or just wait a little longer.")


## Stop the processor

This is the important cell — leaving a stream processor running is what
actually costs money. Stopping preserves its checkpoint for 45 days, so
you can restart it later (`resumeFromCheckpoint`) without losing its place
in the change stream.


In [ ]:
atlas_call("POST", f"/groups/{ATLAS_PROJECT_ID}/streams/{WORKSPACE}/processor/{PROCESSOR}:stop", "2024-05-30")
print("Stopped. Billing for this processor has stopped.")


### Talking points

- Same aggregation syntax as everything else in this repo — `$group`,
  `$match`, `$merge` — just wrapped in a `$source` and a window. Nobody has
  to learn a new query language to build this.
- This is a managed service: no Flink/Kafka Streams cluster to size,
  patch, or scale. Atlas runs the pipeline; you write it.
- Percona Server for MongoDB and Community have change streams (Lab 5
  covers that), but nothing that continuously windows and aggregates them
  for you — you'd be back to hosting your own consumer process and your
  own windowing logic.
- The dead letter queue (`stream_dlq`) is what keeps one malformed document
  from taking the whole pipeline down — check it if a processor's input
  count and output count don't line up the way you expect.


### Optional: tear down the workspace

Only do this if you're done experimenting — it deletes the connection and
workspace along with it. Leaving them in place costs nothing while no
processor is running, so there's no need to run this between sessions.


In [ ]:
# atlas_call("DELETE", f"/groups/{ATLAS_PROJECT_ID}/streams/{WORKSPACE}", "2023-02-01")
# print("Workspace deleted.")
